# 05 — MLflow Datasets: Tracking & Lineage

**UI tab:** Datasets

MLflow 3.x uses `mlflow.genai.datasets.create_dataset()` to create evaluation
datasets that appear in the **Datasets tab**. Each dataset has a stable ID
and you add records with `merge_records()`.

```
eval-questions-v1  (3 easy questions)   →  dataset_id: d-abc...
eval-questions-v2  (5 mixed questions)  →  dataset_id: d-def...
```

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-genai pandas --quiet

In [ ]:
import os
import pandas as pd
from google import genai
from google.genai import types
import mlflow
from mlflow.genai.datasets import create_dataset, search_datasets

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("05-MLflow-Datasets")
experiment_id = mlflow.get_experiment_by_name(
    "05-MLflow-Datasets"
).experiment_id

print("MLflow", mlflow.__version__, "ready")
print(f"Experiment ID: {experiment_id}")

## Dataset V1 — Create and add records

In [ ]:
# create_dataset() registers a named dataset in the MLflow Datasets tab
dataset_v1 = create_dataset(
    name="eval-questions-v1",
    experiment_id=experiment_id,
    tags={"version": "v1", "difficulty": "easy"},
)

# merge_records() adds rows; each row has 'inputs' and 'expectations'
dataset_v1.merge_records([
    {
        "inputs":       {"question": "What is MLflow?"},
        "expectations": {"expected_response": "MLflow is an open-source platform for managing the ML lifecycle."},
    },
    {
        "inputs":       {"question": "What is experiment tracking?"},
        "expectations": {"expected_response": "Experiment tracking records runs, params and metrics for reproducibility."},
    },
    {
        "inputs":       {"question": "What is a model registry?"},
        "expectations": {"expected_response": "A model registry is a central store for versioning ML models."},
    },
])

print(f"Dataset V1 ID : {dataset_v1.dataset_id}")
print(f"Records       : {len(dataset_v1.to_df())}")
print("Open MLflow UI → Datasets tab → eval-questions-v1 should appear.")

## Run against Dataset V1

In [ ]:
def word_overlap(pred, gt):
    p_words = set(pred.lower().split())
    g_words = set(gt.lower().split())
    return len(p_words & g_words) / max(len(g_words), 1)

df = dataset_v1.to_df()

with mlflow.start_run(run_name="run-dataset-v1"):
    mlflow.log_param("dataset_id",   dataset_v1.dataset_id)
    mlflow.log_param("dataset_name", dataset_v1.name)

    scores = []
    for _, row in df.iterrows():
        q  = row["inputs"]["question"]
        gt = row["expectations"]["expected_response"]
        pred = (client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[q],
            config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
        ).text or '').strip()
        scores.append(word_overlap(pred, gt))

    avg = round(sum(scores) / len(scores), 4)
    mlflow.log_metric("avg_word_overlap", avg)
    print(f"V1 — avg word overlap: {avg}")

## Dataset V2 — Add harder questions (different dataset, same lineage)

In [ ]:
dataset_v2 = create_dataset(
    name="eval-questions-v2",
    experiment_id=experiment_id,
    tags={"version": "v2", "difficulty": "mixed"},
)

# V2 contains all V1 rows PLUS 2 harder ones
dataset_v2.merge_records([
    {
        "inputs":       {"question": "What is MLflow?"},
        "expectations": {"expected_response": "MLflow is an open-source platform for managing the ML lifecycle."},
    },
    {
        "inputs":       {"question": "What is experiment tracking?"},
        "expectations": {"expected_response": "Experiment tracking records runs, params and metrics for reproducibility."},
    },
    {
        "inputs":       {"question": "What is a model registry?"},
        "expectations": {"expected_response": "A model registry is a central store for versioning ML models."},
    },
    {
        "inputs":       {"question": "How does MLflow tracing differ from logging?"},
        "expectations": {"expected_response": "Tracing captures the full execution tree; logging records flat key-value metrics."},
    },
    {
        "inputs":       {"question": "What is the MLflow Model Registry staging workflow?"},
        "expectations": {"expected_response": "The registry supports Staging and Production stages with manual promotion."},
    },
])

print(f"V1: {len(dataset_v1.to_df())} records   V2: {len(dataset_v2.to_df())} records")
print(f"V1 ID: {dataset_v1.dataset_id}")
print(f"V2 ID: {dataset_v2.dataset_id}")
print("Datasets tab now shows both eval-questions-v1 and eval-questions-v2.")

In [ ]:
df_v2 = dataset_v2.to_df()

with mlflow.start_run(run_name="run-dataset-v2"):
    mlflow.log_param("dataset_id",   dataset_v2.dataset_id)
    mlflow.log_param("dataset_name", dataset_v2.name)

    scores = []
    for _, row in df_v2.iterrows():
        q  = row["inputs"]["question"]
        gt = row["expectations"]["expected_response"]
        pred = (client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[q],
            config=types.GenerateContentConfig(system_instruction="Answer in 1-2 sentences.")
        ).text or '').strip()
        scores.append(word_overlap(pred, gt))

    avg = round(sum(scores) / len(scores), 4)
    mlflow.log_metric("avg_word_overlap", avg)
    print(f"V2 — avg word overlap: {avg}  (lower — harder questions)")

## Browse all datasets in this experiment

In [ ]:
# search_datasets() lists all datasets linked to this experiment
all_datasets = search_datasets(
    experiment_ids=[experiment_id],
    order_by=["last_update_time DESC"],
)

print(f"Datasets in experiment {experiment_id}:")
for ds in all_datasets:
    df = ds.to_df()
    print(f"  {ds.name:30s} | {len(df):2d} records | {ds.dataset_id}")

## MLflow UI — What to explore
```
Datasets tab
├── eval-questions-v1  →  3 records  →  click to preview inputs/expectations
└── eval-questions-v2  →  5 records  →  includes 2 harder questions

Experiments → Overview
├── run-dataset-v1  →  avg_word_overlap (higher — easy questions)
└── run-dataset-v2  →  avg_word_overlap (lower  — harder questions)
```

---

## All 5 notebooks complete!

| Notebook | Feature | UI Tab |
|---|---|---|
| 01 | Experiments, runs, params, metrics, artifacts | Overview |
| 02 | Auto-tracing Gemini, `@mlflow.trace`, spans | Traces |
| 03 | Chat session grouping via `update_current_trace` | Traces → Sessions |
| 04 | `mlflow.genai.evaluate()` + `@scorer` | Evaluation runs |
| 05 | `create_dataset()` + `merge_records()` | Datasets |